In [2]:
import json
import os
from sklearn.model_selection import KFold
from collections import defaultdict

def coco_kfold_train_val_test(annotation_path, output_dir, k=5, seed=42):
    os.makedirs(output_dir, exist_ok=True)

    with open(annotation_path, "r") as f:
        coco = json.load(f)

    images = coco["images"]
    annotations = coco["annotations"]
    categories = coco["categories"]

    # Map image_id → annotations
    ann_by_image = defaultdict(list)
    for ann in annotations:
        ann_by_image[ann["image_id"]].append(ann)

    image_ids = [img["id"] for img in images]
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    folds = list(kf.split(image_ids))

    for fold_idx in range(k):
        print(f"Creating fold {fold_idx} (train/val/test)...")

        test_indices = folds[fold_idx][1]
        val_indices = folds[(fold_idx + 1) % k][1]

        test_ids = set(image_ids[i] for i in test_indices)
        val_ids = set(image_ids[i] for i in val_indices)
        train_ids = set(image_ids) - test_ids - val_ids

        split_data = {
            "train": {"images": [], "annotations": []},
            "val": {"images": [], "annotations": []},
            "test": {"images": [], "annotations": []},
        }

        for img in images:
            img_id = img["id"]

            if img_id in train_ids:
                split_data["train"]["images"].append(img)
                split_data["train"]["annotations"].extend(ann_by_image[img_id])

            elif img_id in val_ids:
                split_data["val"]["images"].append(img)
                split_data["val"]["annotations"].extend(ann_by_image[img_id])

            elif img_id in test_ids:
                split_data["test"]["images"].append(img)
                split_data["test"]["annotations"].extend(ann_by_image[img_id])

        # Save each split
        for split in ["train", "val", "test"]:
            coco_split = {
                "images": split_data[split]["images"],
                "annotations": split_data[split]["annotations"],
                "categories": categories,
            }

            out_path = os.path.join(output_dir, f"{split}_fold_{fold_idx}.json")
            with open(out_path, "w") as f:
                json.dump(coco_split, f)

        print(
            f"Fold {fold_idx}: "
            f"{len(split_data['train']['images'])} train / "
            f"{len(split_data['val']['images'])} val / "
            f"{len(split_data['test']['images'])} test images"
        )

    print("\nDone! COCO K-fold train/val/test splits created.")


# Example usage
coco_kfold_train_val_test(
    annotation_path="C:\\Users\\dnnxl\\Downloads\\Pineapple Video.v2i.coco\\train\\_annotations.coco.json",
    output_dir="C:\\Users\\dnnxl\\Documents\\GitHub\\vlm-od-agriculture\\src\\dataset\\video_pineapple\\seeds",
    k=5,
)


Creating fold 0 (train/val/test)...
Fold 0: 922 train / 308 val / 308 test images
Creating fold 1 (train/val/test)...
Fold 1: 922 train / 308 val / 308 test images
Creating fold 2 (train/val/test)...
Fold 2: 923 train / 307 val / 308 test images
Creating fold 3 (train/val/test)...
Fold 3: 924 train / 307 val / 307 test images
Creating fold 4 (train/val/test)...
Fold 4: 923 train / 308 val / 307 test images

Done! COCO K-fold train/val/test splits created.
